In [0]:
#**Section 1: Load and inspect**

base = "/Volumes/cs483eviction/raw/cook_county_foreclosures"
path = f"{base}/cook_county_foreclosures_2012.json"

raw = (spark.read.option("multiLine", True)
            .json(path))

print("foreclosure rows:", raw.count())
raw.printSchema()
display(raw.limit(5))


#**Section 2: Clean and derive columns**


from pyspark.sql import functions as F

fc = (raw
      .withColumnRenamed("Recorded Date", "recorded_date")
      .withColumnRenamed("Execution Date", "execution_date")
      .withColumnRenamed("Doc Number", "doc_number")
      .withColumnRenamed("Doc Type", "doc_type")
      .withColumnRenamed("Consideration Amount", "consideration_amount")
      .withColumnRenamed("Location 1", "location_1")
      .withColumn("recorded_date", F.to_timestamp("recorded_date", "MM/dd/yyyy"))
      .withColumn("execution_date",
          F.when(F.year(F.to_timestamp("execution_date", "MM/dd/yyyy")).between(1900, 2100),
               F.to_timestamp("execution_date", "MM/dd/yyyy")))
      .withColumn("pin_clean", F.lpad(F.regexp_replace("Pin", "-", ""), 14, "0"))
      .withColumn("year", F.year("recorded_date"))
      .withColumn("recorded_month", F.date_format("recorded_date", "yyyy-MM"))
      .withColumn("month_start", F.trunc("recorded_date", "month"))
      .withColumn("latitude", F.nullif(F.regexp_extract("location_1", r"\(([^,]+),", 1), F.lit("")).cast("double"))
      .withColumn("longitude", F.nullif(F.regexp_extract("location_1", r",\s*(-?[0-9.]+)\)", 1), F.lit("")).cast("double"))
      .withColumn("has_location",
          F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
      .withColumn("execution_missing", F.col("execution_date").isNull())
      .withColumn("consideration_missing", F.col("consideration_amount").isNull()))

print("rows after dedupe:", fc.count())
display(fc.select("pin_clean", "doc_number", "doc_type", "recorded_date",
                  "month_start", "has_location").limit(20))


#**Section 3: Missing-value check**


missing = fc.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                     for c in fc.columns])
display(missing)


#**Section 4: Summaries**


monthly = (fc.groupBy("year", "recorded_month", "month_start")
             .agg(F.count("*").alias("filings_total"),
                  F.countDistinct("pin_clean").alias("unique_pins"),
                  F.sum(F.col("has_location").cast("int")).alias("geocoded_filings"))
             .withColumn("geocoded_share", F.col("geocoded_filings") / F.col("filings_total"))
             .orderBy("recorded_month"))

by_type = (fc.groupBy("doc_type")
             .agg(F.count("*").alias("filings_total"))
             .orderBy(F.desc("filings_total")))

display(monthly)
display(by_type)


#**Section 5: Save to Delta table**


(fc.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("cs483eviction.default.cook_foreclosures_2012"))

print("saved rows:", spark.table("cs483eviction.default.cook_foreclosures_2012").count())


#**Section 6: Export to JSON**


df = spark.table("cs483eviction.default.cook_foreclosures_2012").toPandas()
df.to_json("/Volumes/cs483eviction/raw/lcbh_raw/cook_foreclosures_2012.json",
           orient="records", lines=True, date_format="iso")
print("exported", len(df), "rows")

# Note: Translated code to Spark using AI such as Microsoft Copilot 